In [3]:
# ============================================================
# 🌱 AI ORGANIC FARMING ADVISOR
# Google Colab - COMPLETE CODE IN ONE CELL
# LLM-based version WITHOUT Embeddings and ChromaDB
# ============================================================

# ============================
# 1. INSTALL REQUIRED LIBRARIES
# ============================

!pip -q install openai gradio

# ============================
# 2. IMPORT LIBRARIES
# ============================

import gradio as gr
from openai import OpenAI
from google.colab import userdata

# ============================
# 3. GET OPENROUTER API KEY
# ============================

api_key = userdata.get("openrouter")

if not api_key:
    raise ValueError(
        "OpenRouter API key not found. "
        "Please add your API key in Google Colab Secrets "
        "with the name: openrouter"
    )

# ============================
# 4. CONFIGURE OPENROUTER CLIENT
# ============================

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

# ============================
# 5. ORGANIC FARMING KNOWLEDGE
# ============================

organic_farming_knowledge = """
You are an AI Organic Farming Advisor.

Use the following organic farming knowledge to help answer farmer questions.

--------------------------------------------------
ORGANIC FERTILIZERS
--------------------------------------------------

Compost is an excellent organic fertilizer made from decomposed
plant and kitchen waste.

Vermicompost is produced using earthworms and is rich in nutrients.

Farmyard manure is made from decomposed animal dung, urine,
bedding material, and farm waste.

Green manure crops can be grown and incorporated into the soil
to improve soil fertility.

Neem cake can be used as an organic soil amendment and may also
help manage certain soil pests.

--------------------------------------------------
NATURAL PEST CONTROL
--------------------------------------------------

Neem oil is commonly used in organic farming to help manage
many insect pests.

Neem-based sprays should be applied according to the product
label and local organic farming guidelines.

Yellow sticky traps can help monitor and reduce populations
of flying insects such as whiteflies and aphids.

Removing severely infected plant parts can help reduce the spread
of some pests and diseases.

Encouraging beneficial insects such as ladybirds and lacewings
can support natural pest control.

Crop rotation and field sanitation are important preventive methods.

--------------------------------------------------
COMPOST PREPARATION
--------------------------------------------------

To prepare compost, collect organic materials such as dry leaves,
vegetable waste, grass clippings, and other suitable plant residues.

Create alternating layers of dry carbon-rich material and green
nitrogen-rich material.

Maintain adequate moisture without making the pile waterlogged.

Turn the compost periodically to provide oxygen and speed decomposition.

Mature compost is generally dark, crumbly, and has an earthy smell.

Avoid adding plastic, chemical waste, diseased plant material,
or other unsuitable materials.

--------------------------------------------------
ORGANIC FARMING PRACTICES
--------------------------------------------------

Crop rotation helps maintain soil health and can reduce the buildup
of certain pests and diseases.

Mulching helps conserve soil moisture, suppress weeds,
and moderate soil temperature.

Drip irrigation can improve water-use efficiency.

Cover crops help protect soil from erosion and may improve
soil organic matter.

Regular soil testing can help farmers understand nutrient requirements.

Integrated Pest Management (IPM) combines monitoring, prevention,
biological control, mechanical methods, and carefully selected
treatments when necessary.

--------------------------------------------------
TOMATO ORGANIC FARMING
--------------------------------------------------

Tomatoes benefit from fertile, well-drained soil enriched
with mature compost.

Mulching can help maintain soil moisture and reduce weed growth.

Crop rotation is recommended to reduce recurring soil-related
pest and disease problems.

Common organic approaches to pest management include monitoring
plants regularly, physical removal, sticky traps, beneficial insects,
and approved organic treatments.

--------------------------------------------------
RICE ORGANIC FARMING
--------------------------------------------------

Organic rice production can use compost, farmyard manure,
and green manure to improve soil fertility.

Weed management may include manual weeding, mechanical methods,
and appropriate water management.

Crop rotation and field sanitation can help manage pest
and disease pressure.

Farmers should follow locally applicable organic certification
standards and agricultural recommendations.

--------------------------------------------------
IMPORTANT ADVISOR RULES
--------------------------------------------------

1. Give practical and easy-to-understand farming advice.
2. Prefer organic and sustainable farming methods.
3. Use the provided knowledge as the primary source.
4. Do not invent unsupported facts.
5. If the knowledge is insufficient, clearly say that more
   information is needed.
6. Explain recommendations step-by-step when appropriate.
7. Mention important precautions when relevant.
8. Do not claim to provide professional agricultural diagnosis.
9. Encourage farmers to consult local agricultural experts for
   crop-specific, pest-specific, disease-specific, or
   region-specific problems.
10. Ask follow-up questions when important information is missing,
    such as crop name, location, soil type, symptoms, or pest details.
"""

# ============================
# 6. AI ORGANIC FARMING ADVISOR
# ============================

def organic_farming_advisor(question):

    # Check empty input
    if not question or not question.strip():
        return "🌱 Please enter your farming question."

    # Create LLM prompt
    prompt = f"""
{organic_farming_knowledge}

--------------------------------------------------
FARMER'S QUESTION
--------------------------------------------------

{question}

--------------------------------------------------
YOUR TASK
--------------------------------------------------

Answer the farmer's question as an AI Organic Farming Advisor.

Provide:
- A clear answer
- Practical steps
- Organic and sustainable solutions
- Important precautions if needed

If the available knowledge is not enough to answer confidently,
say so clearly instead of making up information.

Answer in simple language that a farmer can easily understand.
"""

    try:

        # Call OpenRouter LLM
        response = client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a helpful AI Organic Farming Advisor. "
                        "Give safe, practical, sustainable farming advice."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.3,
            max_tokens=1000
        )

        # Extract response
        answer = response.choices[0].message.content

        return answer

    except Exception as e:
        return f"""
❌ Error while contacting the AI model.

Error details:
{str(e)}

Please check:
1. Your OpenRouter API key
2. Your internet connection
3. The selected model name
4. Your OpenRouter account credits or limits
"""


# ============================
# 7. TEST THE AI ADVISOR
# ============================

test_question = "How can I control whiteflies on my tomato plants organically?"

print("=" * 60)
print("🌱 AI ORGANIC FARMING ADVISOR - TEST")
print("=" * 60)

print("\nFarmer Question:")
print(test_question)

print("\nAI Advisor Answer:")

print(organic_farming_advisor(test_question))


# ============================
# 8. GRADIO USER INTERFACE
# ============================

def advisor_interface(question):
    return organic_farming_advisor(question)


demo = gr.Interface(

    fn=advisor_interface,

    inputs=gr.Textbox(
        label="🌱 Ask Your Farming Question",
        placeholder=(
            "Example: How can I control whiteflies "
            "on my tomato plants organically?"
        ),
        lines=5
    ),

    outputs=gr.Markdown(
        label="AI Organic Farming Advisor"
    ),

    title="🌱 AI Organic Farming Advisor",

    description=(
        "Ask questions about organic fertilizers, "
        "natural pest control, compost preparation, "
        "tomato farming, rice farming, soil fertility, "
        "and sustainable farming practices."
    ),

    examples=[
        ["How can I prepare compost at home?"],
        ["What organic fertilizers can I use for vegetables?"],
        ["How can I control aphids naturally?"],
        ["What are the benefits of crop rotation?"],
        ["How can I grow tomatoes organically?"],
        ["How can I improve soil fertility naturally?"],
        ["How can I control whiteflies on tomato plants?"],
        ["How can I manage weeds in organic rice farming?"]
    ]

)

# ============================
# 9. LAUNCH GRADIO APP
# ============================

demo.launch(
    share=True,
    debug=True
)

🌱 AI ORGANIC FARMING ADVISOR - TEST

Farmer Question:
How can I control whiteflies on my tomato plants organically?

AI Advisor Answer:
To control whiteflies on your tomato plants organically, you can follow these practical steps:

### 1. **Monitor Your Plants**
   - Regularly check your tomato plants for whiteflies. Look under the leaves where they tend to hide. Early detection is key to managing them effectively.

### 2. **Use Yellow Sticky Traps**
   - Place yellow sticky traps around your tomato plants. These traps attract and catch whiteflies and other flying insects, helping to reduce their population.

### 3. **Encourage Beneficial Insects**
   - Introduce or encourage beneficial insects like ladybirds and lacewings in your garden. These insects feed on whiteflies and can help keep their numbers down.

### 4. **Neem Oil Spray**
   - Apply neem oil according to the product label. Neem oil is effective against whiteflies and is safe for your plants. Make sure to spray in the early

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7e5b67e0cffc73de4b.gradio.live
